# Rilevamento vertici faccia cubo — training YOLOv8n-pose

Step 3 del piano "vision": alleniamo un rilevatore dei 4 vertici per ogni faccia visibile del cubo, sul dataset sintetico generato da `web/vision/dataset/generate-dataset.ts` fuso con i fotogrammi reali annotati a mano (`web/vision/annotate/`, via `merge-real-annotations.ts`) — una classe, 4 keypoint, formato Ultralytics pose.

**Prima di eseguire**: `Runtime > Cambia tipo di runtime > GPU (T4 va bene)`.

Questo notebook non fa altro che allenare + esportare: la generazione del dataset, l'annotazione manuale e la fusione dei due sono gia' state fatte e verificate a parte (Step 1-2 e la fusione, vedi `training/README.md`).

**Puoi usare "Esegui tutte le celle" senza problemi**: la scelta della fonte del dataset (`DATASET_SOURCE` sotto) e' un interruttore esplicito — la cella non selezionata si auto-salta invece di eseguire comunque, ed ogni passaggio che fallisce interrompe subito il notebook invece di proseguire in silenzio con un percorso sbagliato.

In [ ]:
!nvidia-smi

## 1. Dataset

**`drive_zip` (consigliata, piu' veloce, unica con i dati reali)**: il dataset e' gia' stato generato in locale (6000 train + 800 val sintetici, fusi con 125 fotogrammi reali annotati a mano — 104 train + 21 val — totale 6104 train + 821 val, ~155MB). Carica `web/vision/dataset/cube-face-keypoints-dataset.zip` in `MyDrive/rubik-vision/` sul tuo Google Drive, poi lascia `DATASET_SOURCE = 'drive_zip'` sotto.

**`regenerate_in_colab`**: rigenera SOLO la parte sintetica direttamente in Colab (reinstalla Node/Chrome ogni sessione — utile solo se vuoi piu' immagini sintetiche o una variazione diversa senza ricaricare lo zip). **Non include i fotogrammi reali annotati** (nessuna fusione in questo ramo): usalo solo per esperimenti sulla parte sintetica, non per il training "buono". Richiede che il branch sia gia' pushato su GitHub: `git push -u origin feat/vision-face-keypoints` dal tuo repo locale, PRIMA di eseguire questa cella.

In [ ]:
DATASET_SOURCE = 'drive_zip'  #@param ["drive_zip", "regenerate_in_colab"]

# usati solo se DATASET_SOURCE == 'drive_zip'
ZIP_PATH = '/content/drive/MyDrive/rubik-vision/cube-face-keypoints-dataset.zip'  #@param {type:"string"}

# usati solo se DATASET_SOURCE == 'regenerate_in_colab'
REPO_URL = 'https://github.com/AndreaAzzarello/rubik-solve-coach.git'  #@param {type:"string"}
BRANCH = 'feat/vision-face-keypoints'  #@param {type:"string"}
TRAIN_COUNT = 6000  #@param {type:"integer"}
VAL_COUNT = 800  #@param {type:"integer"}

In [ ]:
# Unica cella che prepara il dataset: esegue SOLO il ramo di DATASET_SOURCE
# scelto sopra, l'altro si auto-salta. `check=True`/`assert` ovunque: se un
# passaggio fallisce il notebook si ferma qui con un errore chiaro, invece di
# proseguire con DATA_YAML che punta a un percorso mai creato (il bug gia'
# visto: un comando `!` fallito non interrompe da solo l'esecuzione).
import os
import re
import subprocess
import zipfile


def fix_data_yaml_path(yaml_path, dataset_root):
    # Ultralytics risolve un `path:` relativo nel data.yaml rispetto alla
    # working directory del PROCESSO DI TRAINING (qui /content), non rispetto
    # alla cartella dove sta il file - con `path: .` il training cerca
    # /content/images/val invece di {dataset_root}/images/val e fallisce con
    # "images not found" (visto dal vivo). Sovrascriviamo sempre con
    # l'assoluto vero, anche se il file generato lo avesse gia' corretto.
    text = open(yaml_path).read()
    text = re.sub(r'^path:.*$', f'path: {dataset_root}', text, count=1, flags=re.MULTILINE)
    with open(yaml_path, 'w') as handle:
        handle.write(text)


DATA_YAML = None

if DATASET_SOURCE == 'drive_zip':
    from google.colab import drive
    drive.mount('/content/drive')

    DATASET_DIR = '/content/dataset'
    assert os.path.exists(ZIP_PATH), f"Non trovo {ZIP_PATH}: hai caricato lo zip su Drive nel percorso giusto?"
    os.makedirs(DATASET_DIR, exist_ok=True)
    # zipfile della standard library invece del binario `unzip`: niente
    # ambiguita' sui codici di uscita (unzip usa 1 per "warning non fatale,
    # estrazione comunque riuscita", che check=True trattava come un errore
    # fatale). BadZipFile qui vuol dire davvero corrotto, non un falso allarme.
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(DATASET_DIR)
    DATA_YAML = f'{DATASET_DIR}/data.yaml'
    fix_data_yaml_path(DATA_YAML, DATASET_DIR)

elif DATASET_SOURCE == 'regenerate_in_colab':
    NODE_SETUP = '''
set -e
NODE_VERSION=22.13.0
curl -fsSL https://nodejs.org/dist/v${NODE_VERSION}/node-v${NODE_VERSION}-linux-x64.tar.xz -o /tmp/node.tar.xz
mkdir -p /opt/node
tar -xJf /tmp/node.tar.xz -C /opt/node --strip-components=1
ln -sf /opt/node/bin/node /usr/local/bin/node
ln -sf /opt/node/bin/npm /usr/local/bin/npm
ln -sf /opt/node/bin/npx /usr/local/bin/npx
node --version
'''
    subprocess.run(['bash', '-c', NODE_SETUP], check=True)
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, '/content/repo'], check=True)
    subprocess.run(['bash', '-c', 'cd /content/repo/web && npm install -g pnpm && pnpm install --frozen-lockfile'], check=True)
    subprocess.run(['bash', '-c', 'cd /content/repo/web && npx playwright install-deps chromium'], check=True)
    subprocess.run(['bash', '-c', 'cd /content/repo/web && pnpm bench:setup'], check=True)
    subprocess.run(
        ['bash', '-c', f'cd /content/repo/web && node --experimental-strip-types vision/dataset/generate-dataset.ts {TRAIN_COUNT} {VAL_COUNT}'],
        check=True,
    )
    DATASET_DIR = '/content/repo/web/vision/dataset/output'
    DATA_YAML = f'{DATASET_DIR}/data.yaml'
    # Generato in loco nello stesso ambiente in cui si allena: generate-dataset.ts
    # scrive gia' l'assoluto corretto, ma lo riscriviamo comunque per coerenza
    # ed essere robusti a versioni precedenti dello script.
    fix_data_yaml_path(DATA_YAML, DATASET_DIR)

else:
    raise ValueError(f"DATASET_SOURCE sconosciuto: {DATASET_SOURCE!r}")

assert DATA_YAML is not None and os.path.exists(DATA_YAML), f"{DATA_YAML} non esiste: la preparazione del dataset non e' andata a buon fine"
print('DATA_YAML =', DATA_YAML)
print(open(DATA_YAML).read())

## 2. Training

`DATA_YAML` deve gia' esistere (dalla cella sopra).

In [ ]:
MODEL_VARIANT = 'yolov8n-pose.pt'  #@param ["yolov8n-pose.pt", "yolov8s-pose.pt", "yolo11n-pose.pt"]
EPOCHS = 100  #@param {type:"integer"}
IMG_SIZE = 512  #@param {type:"integer"}

assert 'DATA_YAML' in dir() and os.path.exists(DATA_YAML), "Esegui prima la cella '1. Dataset' sopra per definire DATA_YAML"

!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_VARIANT)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=-1,  # batch size automatico in base alla memoria GPU disponibile
    project='runs',
    name='cube_face_keypoints',
)

In [ ]:
# Metriche Ultralytics native (mAP/OKS pose) sul val set. La metrica specifica
# del piano (PCK / grid-cell hit-rate, vedi conversazione) e' il prossimo step,
# non ancora incluso qui.
metrics = model.val()
print(metrics)

## 3. Export ONNX

In [ ]:
onnx_path = model.export(format='onnx', opset=12, simplify=True, imgsz=IMG_SIZE)
print(onnx_path)

## 4. Salva i risultati (scegli una delle due celle)

In [ ]:
# Opzione 1: copia su Google Drive (persiste tra sessioni)
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
DEST = '/content/drive/MyDrive/rubik-vision/models'
os.makedirs(DEST, exist_ok=True)
shutil.copy(onnx_path, DEST)
shutil.copy(str(model.trainer.best), DEST)
print(f'Copiati in {DEST}')

In [ ]:
# Opzione 2: download diretto nel browser
from google.colab import files
files.download(onnx_path)

## Prossimo passo

Il `.onnx` esportato va copiato in `web/vision/models/cube-face-keypoints.onnx` nel repo. Prima di integrarlo nel browser, il piano prevede una verifica di correttezza sul modello stesso: metrica PCK e grid-cell hit-rate sul val set sintetico (Step successivo, non ancora fatto), poi un controllo di trasferimento su qualche frame reale prima di toccare `lib/video-decoder.ts`.